# 03 — The class XGBoost model, ported line by line

**Reads:** `data/raw/prices_yahoo.csv`.

**Writes:** `data/processed/pred_original.csv` (the model's prediction for every stock on every test day),
`results/tables/original_as_scripted.csv` (the performance numbers the class script prints) and
`results/tables/original_as_scripted_monthly.csv` (the two lines of its chart).

This is a Python port of `reference/original_XGBoostStockPrediction.R`: same universe, same start date, same
features, same target, same train/test split, same hyperparameters, same portfolio rule, same performance
formulas. Nothing is corrected here, on purpose. The section numbers follow the R script so the two can be read
side by side. What the script gets wrong is listed at the bottom and fixed in notebooks 04 and 05.

Two differences from running the R script today cannot be avoided: `BK` no longer returns data from Yahoo
Finance (the same company's history under `BNY` is used), and R's and Python's XGBoost builds differ in small
numerical details, so individual predictions will not match to the last decimal.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import xgboost as xgb

ROOT = Path.cwd().parent
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
TAB = ROOT / "results" / "tables"

SCRIPT_START = "2022-01-01"        # the R script's `from`
SPLIT = "2024-01-01"               # train before, test from

## 2–3. Prices and the SPY benchmark, from 2022 as in the script

In [2]:
allpx = pd.read_csv(RAW / "prices_yahoo.csv", parse_dates=["date"])
allpx = allpx[allpx["date"] >= SCRIPT_START]

prices = allpx[~allpx["symbol"].isin(["SPY", "^IRX"])].copy()
spy = (allpx[allpx["symbol"] == "SPY"][["date", "adjusted"]]
       .rename(columns={"adjusted": "spy_price"}).sort_values("date"))
spy["spy_ret"] = spy["spy_price"] / spy["spy_price"].shift(1) - 1
prices["symbol"].nunique(), prices["date"].min().date(), prices["date"].max().date()

(96, datetime.date(2022, 1, 3), datetime.date(2026, 11, 12))

## 4. Features, exactly as written in the script

In R, `lag(x, k)` is the value k rows earlier, which is `shift(k)` here. The script's "momentum" is
`lag(adjusted, 21) / lag(adjusted, 1) - 1`: the price 21 days ago divided by yesterday's price. That is the
older price over the newer one, so it is high when the stock has *fallen*.

In [3]:
df = prices.sort_values(["symbol", "date"]).copy()
g = df.groupby("symbol")["adjusted"]

df["return"] = df["adjusted"] / g.shift(1) - 1
df = df.merge(spy[["date", "spy_ret"]], on="date", how="left")

g = df.groupby("symbol")["adjusted"]
df["mom_1m"] = g.shift(21) / g.shift(1) - 1
df["mom_3m"] = g.shift(63) / g.shift(1) - 1
df["mom_6m"] = g.shift(126) / g.shift(1) - 1
# slide_dbl(lag(return), sd, .before = 21, .complete = TRUE): s.d. of 22 lagged daily returns
df["vol_1m"] = (df.groupby("symbol")["return"]
                .transform(lambda r: r.shift(1).rolling(22, min_periods=22).std()))
df["ret_5d"] = g.shift(5) / g.shift(1) - 1
df["excess_ret"] = df["return"] - df["spy_ret"]
df = df.dropna()

## 5. Target

`lead(adjusted, 20) / lead(adjusted, 1) - 1`: the return from tomorrow's close to the close 20 days ahead,
which is 19 trading days.

In [4]:
g = df.groupby("symbol")["adjusted"]
df["target"] = g.shift(-20) / g.shift(-1) - 1
features_all_days = df.copy()          # kept so notebook 05 can score the latest rebalance dates too
df = df.dropna()

## 6–7. Split and predictors

The recipe is `target ~ .` with only `date` and `symbol` set aside, so **every other column is a predictor**,
including the raw price levels and the share volume.

In [5]:
train = df[df["date"] < SPLIT]
test = df[df["date"] >= SPLIT]
PREDICTORS = [c for c in df.columns if c not in ("symbol", "date", "target")]
print(PREDICTORS)
print(f"train {train['date'].min().date()} to {train['date'].max().date()}, {len(train):,} rows;  "
      f"test {test['date'].min().date()} to {test['date'].max().date()}, {len(test):,} rows")

['open', 'high', 'low', 'close', 'volume', 'adjusted', 'return', 'spy_ret', 'mom_1m', 'mom_3m', 'mom_6m', 'vol_1m', 'ret_5d', 'excess_ret']
train 2022-07-06 to 2023-12-29, 36,000 rows;  test 2024-01-02 to 2026-10-15, 67,200 rows


## 8–10. Model and predictions

`boost_tree(trees = 300, tree_depth = 5, learn_rate = 0.05, loss_reduction = 0, min_n = 10)`. In XGBoost's own
names: 300 rounds, `max_depth` 5, `eta` 0.05, `gamma` 0, `min_child_weight` 10, no row or column subsampling.
With no subsampling the fit has no random element.

In [6]:
ORIGINAL_PARAMS = dict(n_estimators=300, max_depth=5, learning_rate=0.05, gamma=0, min_child_weight=10,
                       subsample=1.0, colsample_bynode=1.0, objective="reg:squarederror", n_jobs=4)
model = xgb.XGBRegressor(**ORIGINAL_PARAMS)
model.fit(train[PREDICTORS], train["target"])

preds = test.copy()
preds["pred"] = model.predict(test[PREDICTORS])

# predictions for every test day, including the last 20 days whose target is not known yet
scored = features_all_days[features_all_days["date"] >= SPLIT].copy()
scored["pred"] = model.predict(scored[PREDICTORS])
scored[["date", "symbol", "pred"]].to_csv(PROC / "pred_original.csv", index=False)

importance = pd.Series(model.feature_importances_, index=PREDICTORS).sort_values(ascending=False)
importance.rename("gain_share").round(3).to_csv(TAB / "original_feature_importance.csv", index_label="feature")
importance.round(3)

mom_3m        0.112
high          0.100
spy_ret       0.098
low           0.089
mom_6m        0.086
vol_1m        0.084
adjusted      0.082
close         0.076
volume        0.066
open          0.064
mom_1m        0.061
ret_5d        0.036
return        0.024
excess_ret    0.022
dtype: float32

## 12–14. The script's backtest

On the first test day of each month: the ten highest predictions, weighted by the prediction (negative ones set
to zero). The portfolio's return for the month is the weighted **target**, the 19-day return above. SPY's return
is computed by calendar month, with the lag taken inside each month, and its cumulative line starts in
January 2022.

In [7]:
bt = preds.copy()
bt["month"] = bt["date"].dt.to_period("M").dt.to_timestamp()
bt = bt[bt["date"] == bt.groupby("month")["date"].transform("min")]
bt = bt.sort_values(["month", "pred"], ascending=[True, False]).groupby("month").head(10).copy()
bt["pred_positive"] = bt["pred"].clip(lower=0)
tot = bt.groupby("month")["pred_positive"].transform("sum")
bt["weight"] = np.where(tot == 0, 0.1, bt["pred_positive"] / tot.replace(0, np.nan))

returns = (bt.assign(wr=bt["weight"] * bt["target"])
           .groupby(["month", "date"], as_index=False)["wr"].sum()
           .rename(columns={"wr": "portfolio_return"}))
returns["cumulative_return"] = (1 + returns["portfolio_return"]).cumprod()

s = spy.copy()
s["month"] = s["date"].dt.to_period("M").dt.to_timestamp()
s["r_within"] = s["spy_price"] / s.groupby("month")["spy_price"].shift(1) - 1     # lag inside the month
spy_returns = (s.groupby("month")["r_within"].apply(lambda r: (1 + r.dropna()).prod() - 1)
               .rename("spy_return").reset_index())
spy_returns["spy_cumulative"] = (1 + spy_returns["spy_return"]).cumprod()            # starts in 2022
returns = returns.merge(spy_returns, on="month", how="left")
returns.round(4).to_csv(TAB / "original_as_scripted_monthly.csv", index=False)
returns.tail(3).round(3)

,month,date,portfolio_return,cumulative_return,spy_return,spy_cumulative
31,2026-08-01,2026-08-03,0.008,2.304,0.012,1.657
32,2026-09-01,2026-09-01,0.037,2.389,0.004,1.663
33,2026-10-01,2026-10-01,0.050,2.508,-0.011,1.645


## 15. The script's performance numbers

The script multiplies by √252 and by 252, the factors for **daily** returns, although `portfolio_return` is one
number per **month**. The table reports the script's figures next to the same figures with the monthly
factors (√12 and 12).

In [8]:
pr, sr = returns["portfolio_return"], returns["spy_return"]
beta, alpha_m = np.polyfit(sr, pr, 1)

as_scripted = pd.Series({
    "months": len(returns),
    "first_month": returns["month"].min().date().isoformat(),
    "last_month": returns["month"].max().date().isoformat(),
    "sharpe_as_scripted_sqrt252": pr.mean() / pr.std(ddof=1) * np.sqrt(252),
    "sharpe_monthly_factor_sqrt12": pr.mean() / pr.std(ddof=1) * np.sqrt(12),
    "alpha_as_scripted_x252": alpha_m * 252,
    "alpha_monthly_factor_x12": alpha_m * 12,
    "beta": beta,
    "ml_growth_of_1_end": returns["cumulative_return"].iloc[-1],
    "spy_line_start": returns["spy_cumulative"].iloc[0],
    "spy_line_end": returns["spy_cumulative"].iloc[-1],
    "spy_growth_over_test_months": returns["spy_cumulative"].iloc[-1] / returns["spy_cumulative"].iloc[0]
                                   * (1 + returns["spy_return"].iloc[0]),
    "mean_weight_of_largest_holding": bt.groupby("month")["weight"].max().mean(),
    "train_rows": len(train), "test_rows": len(test), "predictors": len(PREDICTORS),
})
as_scripted.rename("value").to_csv(TAB / "original_as_scripted.csv", index_label="statistic")
as_scripted

months                                    34
first_month                       2024-01-01
last_month                        2026-10-01
sharpe_as_scripted_sqrt252          8.835654
sharpe_monthly_factor_sqrt12        1.928098
alpha_as_scripted_x252              3.301302
alpha_monthly_factor_x12            0.157205
beta                                0.966491
ml_growth_of_1_end                   2.50805
spy_line_start                      0.992841
spy_line_end                        1.644925
spy_growth_over_test_months         1.692645
mean_weight_of_largest_holding      0.186494
train_rows                             36000
test_rows                              67200
predictors                                14
dtype: object

## What the script does that the next notebooks change

1. **Raw price levels and share volume are predictors.** A share price identifies the company more than it
   describes its prospects, and the levels in the test period lie outside the range seen in training.
2. **The momentum features are inverted** (older price over newer price).
3. **The target is the raw return**, most of which is the market's move, the same for every stock that day.
   The portfolio only needs to know which stocks will do better than the others.
4. **Eighteen months of training data** (mid-2022 to end-2023), fitted once and never updated.
5. **Training targets reach into the test period**: a row from late December 2023 has a target that ends in
   late January 2024.
6. **The backtest compares unlike things**: a 19-day return that starts on the second trading day of the month
   against SPY's calendar-month return, with SPY's line started two years earlier.
7. **The Sharpe ratio and alpha are annualised with daily factors** applied to monthly numbers, which inflates
   the Sharpe ratio by √(252/12) ≈ 4.6 and alpha by 21.
8. **Weights proportional to the prediction** concentrate the portfolio in one or two names.
9. **The universe was chosen in 2026**: these are companies known to be large today.